Schema

 - Matches:
     - MatchId: int
         - MatchWeek: int
         - HomeTeam: int
         - AwayTeam: int
         - MatchResult: pt.MatchResult
         - ThreadUrl: str
         - MatchTime: datetime (convert to UTC?)

 - Standings:
     - MatchId: int
         - UserName: str
             - Points: int

 - Predictions:
     - MatchId: int
         - UserName: str
             - Comment: str
             - CommentTime: datetime (convert to UTC?)


What if someone posts two comments?

{
    "MatchId": {
        "0": {
            "MatchWeek": 0,
            "HomeTeam": "Southampton",
            "
        }
    }
}

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
from icecream import ic

import predthread as pt
from predthread import fbref

from collections import defaultdict
from copy import deepcopy

In [3]:
root = Path(".").absolute()
print(root)

/Users/julianirwin/Projects/predthread/examples/southamton_24-25_33c895d4


In [7]:
club_names = pt.club_names(root)
api_keys = pt.api_keys(root)
match_metadata = pt.match_metadata(root, match_id=0)
matches_metadata = pt.matches_metadata(root)
matches_metadata_df = pt.matches_metadata_df(root)

# ic(club_names)
# ic(match_metadata)
# ic(matches_metadata)

In [5]:
# Of last completed match
match_id = 7

In [8]:
matches_metadata_df

,MatchWeek,HomeClub,AwayClub,HomeGoals,AwayGoals,MatchTimeUtc,ThreadUrl
0,1,Newcastle Utd,Southampton,1,0,2024-08-17T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1et...
1,2,Southampton,Nott'ham Forest,0,1,2024-08-24T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1ey...
2,3,Brentford,Southampton,3,1,2024-08-31T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1f4...
3,4,Southampton,Manchester Utd,0,3,2024-09-14T12:30:00Z,https://www.reddit.com/r/SaintsFC/comments/1ff...
4,5,Southampton,Ipswich Town,1,1,2024-09-21T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1fk...
5,6,Bournemouth,Southampton,3,1,2024-09-30T20:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1fr...
6,7,Arsenal,Southampton,3,1,2024-10-05T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1fv...
7,8,Southampton,Leicester City,2,3,2024-10-19T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1g6...
8,9,Manchester City,Southampton,1,0,2024-10-26T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1gb...
9,10,Southampton,Everton,1,0,2024-11-02T15:00:00Z,https://www.reddit.com/r/SaintsFC/comments/1gh...


In [14]:
def _next_match_id(match_metadata: dict):
    try:
        return max(map(int, match_metadata["MatchId"].keys()))
    except ValueError:
        return "0"

'9'

In [15]:
# Using matchweek, not match id
i_matchweek = 11
ic(fbref.match_metadata(i_matchweek=i_matchweek))
# pt.append_match_metadata(root, fbref.match_metadata(i_matchweek))

ic| fbref.match_metadata(i_matchweek=i_matchweek): {'AwayClub': 'Southampton',
                                                    'AwayGoals': '',
                                                    'HomeClub': 'Wolves',
                                                    'HomeGoals': '',
                                                    'MatchTimeUtc': '2024-11-09T15:00:00Z',
                                                    'MatchWeek': 11,
                                                    'ThreadUrl': ''}


TypeError: 'DataFrame' object is not callable

In [11]:
pd.set_option("display.max_rows", None)
ps = pt.download_predictions(root, match_id)
ps.sort_values(["PredictedHomeGoals", "PredictedAwayGoals"], ascending=False)
# ps

,Comment,PredictedHomeGoals,PredictedAwayGoals
UserName,,,
Antidote-Killer,11-4,11,4
nape27,3-3,3,3
master_shifwho,3-3,3,3
Holy_Shit_Balls_69,3-2,3,2
BJH19,3-2,3,2
allbarneynorubble,3-1,3,1
Relevant_Rev,3-1,3,1
whowantstogo,3-0,3,0
rghsfc,3-0 defeat,3,0


In [12]:
pt.summarize_predictions(root, match_id)

{'TotalPredictions': 88,
 'TotalHomeWinPredictions': 31,
 'TotalAwayWinPredictions': 36,
 'TotalDrawPredictions': 21}

In [13]:
new_standings = pt.calculate_updated_standings_after(root, match_id)
new_standings

,Points,PointsGained,Streak,Exacts,Corrects,Wrongs,MaxStreak
UserName,,,,,,,
Nixstricks,11,0,0,3,2,3,7
LegitimatePass6924,10,1,5,2,4,2,5
flailingpariah,9,1,3,1,6,1,6
slugmaniac,9,0,0,2,3,3,4
Saint_Noog,8,1,7,1,5,1,7
jjbc56,8,1,7,1,5,2,7
halarhala,8,1,3,1,5,1,5
Nas1729,8,1,3,1,5,2,4
Zapdosman,8,1,2,1,5,2,5


In [14]:
print(new_standings.reset_index().to_markdown())

|     | UserName             |   Points |   PointsGained |   Streak |   Exacts |   Corrects |   Wrongs |   MaxStreak |
|----:|:---------------------|---------:|---------------:|---------:|---------:|-----------:|---------:|------------:|
|   0 | Nixstricks           |       11 |              0 |        0 |        3 |          2 |        3 |           7 |
|   1 | LegitimatePass6924   |       10 |              1 |        5 |        2 |          4 |        2 |           5 |
|   2 | flailingpariah       |        9 |              1 |        3 |        1 |          6 |        1 |           6 |
|   3 | slugmaniac           |        9 |              0 |        0 |        2 |          3 |        3 |           4 |
|   4 | Saint_Noog           |        8 |              1 |        7 |        1 |          5 |        1 |           7 |
|   5 | jjbc56               |        8 |              1 |        7 |        1 |          5 |        2 |           7 |
|   6 | halarhala            |        8 |       

In [ ]:
predictions = pt.load_predictions(root, match_id)
old_standings = pt.load_standings_after(root, match_id - 1)
new_standings_p = pt.load_standings_after(root, match_id)
new_standings_p = new_standings.rename(columns=lambda x: x + "*")
predictions.join(old_standings, new_standings_p)

In [15]:
pt.summarize_standings_after(root, match_id)

{'TotalExacts': 0, 'TotalCorrects': 36}

# Post Template

## Todo

 - Add callouts for exacts and streaks

In [19]:
m_next = pt.match_metadata(root, match_id+1)
m_last = pt.match_metadata(root, match_id)

s_pred = pt.summarize_predictions(root, match_id - 1)
s_stand = pt.summarize_standings_after(root, match_id-1)

print(f"""
# Prediction Thread Match {match_id + 2}: {m_next["HomeClub"]} vs {m_next["AwayClub"]}

# Home - Away Format

 - Home: {m_next["HomeClub"]}
 - Away: {m_next["AwayClub"]}

# Game Rules

 - Post a top level comment in this thread at least one hour before kickoff, before lineups are posted.
 - Follow the format of this example comment: "0 - 3. Banter goes here."
 - Comment format must be "[Home Score] - [Away Score]. [Optional explanation, detailed prediction, or general shit talking here.]"
 - Scoring:
    - **3 points** for an exactly correct prediction.
    - **1 points** for a correct result.
    - **0 points** for an incorrect result.
- DM me with any problems, questions, comments.

# Summary of Last Match ({m_last["HomeClub"]} - {m_last["AwayClub"]})

{m_last["ThreadUrl"]}

 - Total Predictions: {s_pred["TotalPredictions"]}
 - Predicted Home Wins: {s_pred["TotalHomeWinPredictions"]} 
 - Predicted Away Wins: {s_pred["TotalAwayWinPredictions"]} 
 - Predicted Draws: {s_pred["TotalDrawPredictions"]}
 - Exacts: {s_stand["TotalExacts"]}
 - Corrects: {s_stand["TotalCorrects"]} 

# Standings
{new_standings.reset_index().to_markdown()}
""")


# Prediction Thread Match 9: Manchester City vs Southampton

# Home - Away Format

 - Home: Manchester City
 - Away: Southampton

# Game Rules

 - Post a top level comment in this thread at least one hour before kickoff, before lineups are posted.
 - Follow the format of this example comment: "0 - 3. Banter goes here."
 - Comment format must be "[Home Score] - [Away Score]. [Optional explanation, detailed prediction, or general shit talking here.]"
 - Scoring:
    - **3 points** for an exactly correct prediction.
    - **1 points** for a correct result.
    - **0 points** for an incorrect result.
- DM me with any problems, questions, comments.

# Summary of Last Match (Southampton - Leicester City)

https://www.reddit.com/r/SaintsFC/comments/1g64wgc/prediction_thread_match_8_southampton_vs/

 - Total Predictions: 108
 - Predicted Home Wins: 97 
 - Predicted Away Wins: 9 
 - Predicted Draws: 2
 - Exacts: 3
 - Corrects: 94 

# Standings
|     | UserName             |   Points |   Points